# N6: QLoRA SFT + DPO Alignment (Qwen/Qwen2.5-7B-Instruct / Airavata)

- **Base Model**: `Qwen/Qwen2.5-7B-Instruct`
- **Dataset**: 5,000+ trilingual (te/hi/en) grounded explanation pairs + DPO chosen (rule-faithful) vs rejected (hallucinated verdict) pairs
- **Objective**: 4-bit NF4 QLoRA SFT (r=16, alpha=32) followed by DPO (beta=0.1)
- **Target**: Win rate >= 65% vs base model on 200 prompts (LLM-as-judge) with 0 verdict overrides
- **Seed**: `42` (pinned across Python, NumPy, PyTorch)
- **Hardware**: Kaggle / Colab T4 (16 GB VRAM). Set `SAMPLE_FRAC=0.05` for top-to-bottom smoke test.


In [ ]:
# Pinned versions for reproducible Kaggle T4 execution
!pip install -q transformers==4.47.1 sentence-transformers==3.3.1 peft==0.14.0 trl==0.13.0 bitsandbytes==0.45.0 mlflow==2.19.0 huggingface_hub==0.27.0


In [ ]:
import os, random, torch
from transformers import BitsAndBytesConfig
from peft import LoraConfig

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
print("Configured QLoRA NF4 + DPO (beta=0.1, seed=42) -> checkpoints/N6_qlora_dpo")


In [ ]:
# MLflow logging + Hugging Face Hub push (requires HF_TOKEN env var)
import os, mlflow
mlflow.set_experiment('yojana-mitra-training')
with mlflow.start_run(run_name='N6_qlora_dpo'):
    mlflow.log_params({'base_model': 'Qwen/Qwen2.5-7B-Instruct', 'seed': 42, 'sample_frac': os.getenv('SAMPLE_FRAC', '0.05')})
if os.getenv('HF_TOKEN'):
    from huggingface_hub import HfApi
    print('Ready to upload checkpoints/N6_qlora_dpo to yojana-mitra/qwen25-7b-welfare-explainer-dpo')
